# 记忆工程完整串行流程演示

## Objective / 目标
- 演示记忆系统的**完整串行流程**
- 正确的流程：**抽取 → 更新（查询相关记忆决定操作） → 存储（Neo4j + Milvus同步） → 组织 → 检索回答问题**

## 核心流程

```
对话输入 
    ↓
[1. 记忆抽取] - 从对话中抽取候选记忆
    ↓
[2. 更新决策] - 查询已有记忆，决定：新增/合并/覆盖/冲突处理
    ↓
[3. 统一存储] - Neo4j(结构化) + Milvus(向量) 同步写入，ID对应
    ↓
[4. 记忆组织] - 构建时间/事件/语义/层级结构
    ↓
[5. 检索回答] - 根据用户问题检索记忆，注入上下文，LLM回答
```

In [1]:
# Setup: environment & reproducibility / 环境与可复现
import sys
import os

# 添加项目根目录到路径（这样可以 import src）
sys.path.insert(0, os.path.dirname(os.getcwd()))

print("Python:", sys.version.split()[0])
print("Working directory:", os.getcwd())

Python: 3.10.18
Working directory: /Users/chunyuli/Documents/osworkspace/MemFactory/example


In [2]:
# Imports / 依赖集中声明
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional, Tuple

# 从 src 包统一导入所有记忆工程模块
from src import (
    # 数据结构
    MemoryItem, ConversationMessage, ExtractionResult, SearchResult, Edge,
    
    # 枚举类型
    MemoryType, MemoryStatus, UpdateAction, RelationType,
    
    # 客户端
    LLMClient, EmbeddingClient, Neo4jClient, MilvusClient, MemoryStore,
    get_llm_client, get_embedding_client, get_neo4j_client, get_milvus_client, get_memory_store,
    
    # 工具函数
    generate_id, current_timestamp, format_conversation,
    
    # 配置类
    ExtractionConfig, SearchConfig, OrganizationConfig, UpdateConfig,
    
    # 主要类
    MemoryExtractor, MemorySearcher, MemoryOrganizer, MemoryUpdater,
)

print("所有模块导入成功！")

所有模块导入成功！


In [3]:
# Config / 配置
USER_ID = "demo_user"
SESSION_ID = "demo_session"
VERBOSE = True

print(f"用户ID: {USER_ID}")
print(f"会话ID: {SESSION_ID}")

用户ID: demo_user
会话ID: demo_session


## Step 1: 准备测试数据

构造一组模拟对话，用于演示完整的记忆处理流程

In [4]:
# 构造示例对话 - 第一轮：建立基础记忆
sample_conversation = [
    ConversationMessage(
        role="user",
        content="我是Tom，最近在负责一个AI项目，团队有5个人。",
        timestamp="2025-06-26T10:00:00"
    ),
    ConversationMessage(
        role="assistant",
        content="你好Tom！听起来是个很有意思的项目。项目进展如何？",
        timestamp="2025-06-26T10:01:00"
    ),
    ConversationMessage(
        role="user",
        content="进展还不错，昨天完成了数据预处理模块。不过我有点担心12月15日的截止日期。",
        timestamp="2025-06-26T10:02:00"
    ),
    ConversationMessage(
        role="assistant",
        content="时间确实比较紧。你们团队的分工是怎样的？",
        timestamp="2025-06-26T10:03:00"
    ),
    ConversationMessage(
        role="user",
        content="我负责整体架构和模型训练，小李负责数据，小王负责前端。对了，我平时喜欢喝美式咖啡，不加糖。",
        timestamp="2025-06-26T10:05:00"
    ),
]

print(f"准备了 {len(sample_conversation)} 条对话消息（第一轮：建立基础记忆）")
for msg in sample_conversation:
    print(f"  [{msg.role}] {msg.content[:50]}...")

准备了 5 条对话消息（第一轮：建立基础记忆）
  [user] 我是Tom，最近在负责一个AI项目，团队有5个人。...
  [assistant] 你好Tom！听起来是个很有意思的项目。项目进展如何？...
  [user] 进展还不错，昨天完成了数据预处理模块。不过我有点担心12月15日的截止日期。...
  [assistant] 时间确实比较紧。你们团队的分工是怎样的？...
  [user] 我负责整体架构和模型训练，小李负责数据，小王负责前端。对了，我平时喜欢喝美式咖啡，不加糖。...


## Step 2: 完整Pipeline类定义

实现正确的串行流程：抽取 → 更新决策 → 存储 → 组织 → 检索回答

In [5]:
class MemoryPipelineRunner:
    """
    记忆工程完整串行Pipeline
    
    正确流程：
    1. 记忆抽取 - 从对话中抽取候选记忆
    2. 更新决策 - 查询已有记忆，决定操作（新增/合并/覆盖/冲突处理）
    3. 统一存储 - Neo4j(结构化) + Milvus(向量) 同步写入，ID对应
    4. 记忆组织 - 构建时间/事件/语义/层级结构
    5. 检索回答 - 根据用户问题检索记忆，注入上下文，LLM回答
    """
    
    def __init__(self, user_id: str = "default_user", verbose: bool = True):
        self.user_id = user_id
        self.verbose = verbose
        
        # 初始化统一存储管理器
        self.store = get_memory_store()
        
        # 初始化各模块（不自动保存，由Pipeline统一管理）
        self.extractor = MemoryExtractor(ExtractionConfig(
            strategy="simple",
            auto_save=False,  # 关闭自动保存，由Pipeline控制
            auto_embed=False,
            verbose=verbose,
            user_id=user_id
        ))
        
        self.updater = MemoryUpdater(UpdateConfig(
            strategy="auto",
            auto_save=False,  # 关闭自动保存
            verbose=verbose
        ))
        
        self.organizer = MemoryOrganizer(OrganizationConfig(
            use_temporal=True,
            use_event=True,
            use_semantic=True,
            use_hierarchy=False,  # 简化演示
            auto_save_edges=True,
            verbose=verbose
        ))
        
        self.searcher = MemorySearcher(SearchConfig(
            strategy="passive",
            top_k=5,
            similarity_threshold=0.3,
            verbose=verbose,
            user_id=user_id
        ))
        
        # LLM客户端用于最终回答
        self.llm = get_llm_client()
        
        print(f"[MemoryPipelineRunner] 初始化完成，用户: {user_id}")
    
    def _log(self, stage: str, message: str):
        """日志输出"""
        if self.verbose:
            print(f"[{stage}] {message}")
    
    def process_conversation(self, messages: List[ConversationMessage]) -> Dict[str, Any]:
        """
        处理对话的完整流程
        
        流程: 抽取 → 更新决策 → 存储 → 组织
        
        Args:
            messages: 对话消息列表
            
        Returns:
            处理结果字典
        """
        result = {
            "extracted_memories": [],
            "update_decisions": [],
            "saved_memories": [],
            "graph": None,
            "errors": []
        }
        
        # ========== Step 1: 记忆抽取 ==========
        self._log("Step1-抽取", f"开始从 {len(messages)} 条消息中抽取记忆...")
        
        extraction_result = self.extractor.run(messages)
        
        if extraction_result.status != "SUCCESS":
            self._log("Step1-抽取", f"抽取失败: {extraction_result.message}")
            result["errors"].append(f"抽取失败: {extraction_result.message}")
            return result
        
        candidate_memories = extraction_result.memory_list
        self._log("Step1-抽取", f"抽取到 {len(candidate_memories)} 条候选记忆")
        result["extracted_memories"] = candidate_memories
        
        if not candidate_memories:
            self._log("Step1-抽取", "没有抽取到记忆，流程结束")
            return result
        
        # ========== Step 2: 更新决策（查询相关记忆） ==========
        self._log("Step2-更新决策", "开始对每条候选记忆进行更新决策...")
        
        memories_to_save = []
        
        for i, candidate in enumerate(candidate_memories, 1):
            self._log("Step2-更新决策", f"处理候选记忆 {i}/{len(candidate_memories)}: {candidate.key}")
            
            # 查询相关的已有记忆
            related_memories = self.store.find_related_memories(candidate, top_k=20)
            
            if not related_memories:
                # 没有相关记忆，直接新增
                self._log("Step2-更新决策", f"  → 无相关记忆，决定: ADD")
                decision = {
                    "candidate": candidate,
                    "action": "ADD",
                    "related": [],
                    "final_memory": candidate
                }
            else:
                # 有相关记忆，调用更新模块决策
                self._log("Step2-更新决策", f"  → 找到 {len(related_memories)} 条相关记忆")
                
                for rm, score in related_memories:
                    self._log("Step2-更新决策", f"     - [{score:.3f}] {rm.key}: {rm.value[:50]}...")
                
                # 使用更新模块决定操作
                update_result = self.updater.decide_action(
                    new_memory=candidate,
                    related_memories=[m for m, s in related_memories]
                )
                
                self._log("Step2-更新决策", f"  → 决定: {update_result['action']}")
                
                decision = {
                    "candidate": candidate,
                    "action": update_result["action"],
                    "related": related_memories,
                    "final_memory": update_result.get("final_memory", candidate),
                    "deprecated_ids": update_result.get("deprecated_ids", [])
                }
                
                # 如果是合并，处理被废弃的记忆
                if decision["action"] == "MERGE" and decision.get("deprecated_ids"):
                    for dep_id in decision["deprecated_ids"]:
                        self._log("Step2-更新决策", f"  → 废弃旧记忆: {dep_id}")
            
            result["update_decisions"].append(decision)
            
            # 收集需要保存的记忆
            if decision["action"] in ["ADD", "MERGE", "OVERWRITE"]:
                memories_to_save.append(decision["final_memory"])
        
        # ========== Step 3: 统一存储（Neo4j + Milvus同步） ==========
        self._log("Step3-存储", f"开始保存 {len(memories_to_save)} 条记忆到Neo4j和Milvus...")
        
        for memory in memories_to_save:
            # 确保设置用户ID
            memory.user_id = self.user_id
            
            # 统一保存（自动生成embedding，同步写入Neo4j和Milvus）
            success = self.store.save(memory, generate_embedding=True)
            
            if success:
                self._log("Step3-存储", f"  ✓ 保存成功: {memory.id} - {memory.key}")
                result["saved_memories"].append(memory)
            else:
                self._log("Step3-存储", f"  ✗ 保存失败: {memory.key}")
                result["errors"].append(f"保存失败: {memory.key}")
        
        # ========== Step 4: 记忆组织 ==========
        if result["saved_memories"]:
            self._log("Step4-组织", f"开始组织 {len(result['saved_memories'])} 条记忆...")
            
            graph = self.organizer.run(result["saved_memories"])
            result["graph"] = graph
            
            self._log("Step4-组织", f"  → 构建图谱: {graph.get_node_count()} 节点, {graph.get_edge_count()} 边")
        
        self._log("完成", "对话处理完成！")
        return result
    
    def answer_question(self, question: str) -> Dict[str, Any]:
        """
        根据用户问题检索记忆并回答
        
        流程: 检索相关记忆 → 注入上下文 → LLM生成回答
        
        Args:
            question: 用户问题
            
        Returns:
            回答结果字典
        """
        result = {
            "question": question,
            "retrieved_memories": [],
            "context": "",
            "answer": "",
            "status": "SUCCESS"
        }
        
        # ========== Step 5: 检索相关记忆 ==========
        self._log("Step5-检索", f"检索与问题相关的记忆: {question}")
        
        # 使用统一存储的检索功能
        search_results = self.store.search_similar(
            query=question,
            top_k=5,
            user_id=self.user_id
        )
        
        if not search_results:
            self._log("Step5-检索", "  → 未找到相关记忆")
            result["answer"] = "抱歉，我没有找到相关的记忆信息来回答这个问题。"
            return result
        
        self._log("Step5-检索", f"  → 找到 {len(search_results)} 条相关记忆:")
        
        for memory, score in search_results:
            self._log("Step5-检索", f"     - [{score:.3f}] {memory.key}: {memory.value[:50]}...")
            result["retrieved_memories"].append({
                "memory": memory,
                "score": score
            })
        
        # ========== 构建上下文 ==========
        context_parts = ["以下是与问题相关的记忆信息：\n"]
        for i, (memory, score) in enumerate(search_results, 1):
            context_parts.append(f"{i}. [{memory.key}] {memory.value}")
        
        context = "\n".join(context_parts)
        result["context"] = context
        
        self._log("Step5-检索", "  → 构建上下文完成")
        
        # ========== LLM生成回答 ==========
        self._log("Step5-回答", "调用LLM生成回答...")
        
        system_prompt = "你是一个智能助手，根据提供的记忆信息准确回答用户问题。"
        
        user_prompt = f"""基于以下记忆信息回答用户的问题。请直接、准确地回答，如果记忆中有明确的信息就使用它。

{context}

用户问题: {question}

请用中文回答："""
        
        answer = self.llm.chat(system_prompt, user_prompt)
        result["answer"] = answer
        
        self._log("Step5-回答", f"  → 回答: {answer[:100] if answer else '(空)'}...")
        
        return result
    
    def run_full_pipeline(self, messages: List[ConversationMessage], 
                          questions: List[str]) -> Dict[str, Any]:
        """
        运行完整的端到端流程
        
        1. 处理对话（抽取 → 更新决策 → 存储 → 组织）
        2. 回答问题（检索 → 注入上下文 → LLM回答）
        
        Args:
            messages: 对话消息列表
            questions: 要回答的问题列表
            
        Returns:
            完整结果字典
        """
        print("\n" + "=" * 70)
        print("🚀 开始运行完整记忆工程Pipeline")
        print("=" * 70)
        
        full_result = {
            "conversation_result": None,
            "question_answers": []
        }
        
        # Part 1: 处理对话
        print("\n📝 Part 1: 处理对话 (抽取 → 更新决策 → 存储 → 组织)")
        print("-" * 70)
        
        conv_result = self.process_conversation(messages)
        full_result["conversation_result"] = conv_result
        
        # Part 2: 回答问题
        print("\n\n❓ Part 2: 检索记忆并回答问题")
        print("-" * 70)
        
        for q in questions:
            print(f"\n问题: {q}")
            qa_result = self.answer_question(q)
            full_result["question_answers"].append(qa_result)
            print(f"回答: {qa_result['answer']}")
        
        print("\n" + "=" * 70)
        print("✅ Pipeline运行完成！")
        print("=" * 70)
        
        return full_result


print("MemoryPipelineRunner 类定义完成！")

MemoryPipelineRunner 类定义完成！


In [6]:
# 创建Pipeline实例
pipeline = MemoryPipelineRunner(user_id=USER_ID, verbose=VERBOSE)

[Neo4jClient] 已连接到 bolt://47.117.41.207:7687, 数据库: neo4j
[EmbeddingClient] 已初始化，模型: bge-m3, 服务地址: https://apigw.memtensor.cn/model/embedding/v1


/opt/anaconda3/envs/memos_new/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.30.2 at schema.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/anaconda3/envs/memos_new/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.30.2 at common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/anaconda3/envs/memos_new/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.30.2 at milvus.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/

[MilvusClient] 已连接到 http://c-c31c787e871ed9c9.milvus.aliyuncs.com:19530
[MemoryStore] 运行在真实数据库模式
[MemoryStore] 统一存储管理器初始化完成
[LLMClient] 已初始化，模型: gpt-4.1-nano
[MemoryExtractor] 初始化完成，策略: simple
[MemoryUpdater] 初始化完成，策略: auto
[MemoryOrganizer] 初始化完成
[MemorySearcher] 初始化完成，策略: passive
[MemoryPipelineRunner] 初始化完成，用户: demo_user


## Step 3: 运行完整Pipeline

执行完整流程：抽取 → 更新决策 → 存储 → 组织 → 检索回答

In [7]:
# 定义要回答的问题
questions = [
    "Tom负责什么工作？",
    "项目的截止日期是什么时候？",
    "用户喜欢喝什么饮料？",
    "团队有多少人？"
]

print(f"准备回答 {len(questions)} 个问题:")
for i, q in enumerate(questions, 1):
    print(f"  {i}. {q}")

准备回答 4 个问题:
  1. Tom负责什么工作？
  2. 项目的截止日期是什么时候？
  3. 用户喜欢喝什么饮料？
  4. 团队有多少人？


In [8]:
# 运行完整Pipeline
full_result = pipeline.run_full_pipeline(
    messages=sample_conversation,
    questions=questions
)


🚀 开始运行完整记忆工程Pipeline

📝 Part 1: 处理对话 (抽取 → 更新决策 → 存储 → 组织)
----------------------------------------------------------------------
[Step1-抽取] 开始从 5 条消息中抽取记忆...
[SimpleExtractor] 抽取了 3 条记忆
[Step1-抽取] 抽取到 3 条候选记忆
[Step2-更新决策] 开始对每条候选记忆进行更新决策...
[Step2-更新决策] 处理候选记忆 1/3: 负责AI项目
[Step2-更新决策]   → 找到 20 条相关记忆
[Step2-更新决策]      - [0.080] Tom的AI项目管理: 用户Tom目前负责一个由五人团队合作的AI项目，项目于2025年6月26日进行中。用户在2025年6...
[Step2-更新决策]      - [0.076] 用户担心项目截止日期: 截至2026年1月，用户负责的AI项目进展良好，团队已于2025年6月25日完成数据预处理模块，但用...
[Step2-更新决策]      - [0.054] AI项目进展: 用户Tom最近负责一个AI项目，团队共有5人。用户已经完成了数据预处理模块，并且担心12月15日的截...
[Step2-更新决策]      - [0.053] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练，团队共有五人，项目截止日期为2025年12月15日。...
[Step2-更新决策]      - [0.049] 用户团队成员分工: 在2026年1月20日，用户Tom负责整体架构和模型训练，小李负责数据，小王负责前端。到2026年1...
[Step2-更新决策]      - [0.048] 个人喜好变化: 用户现在更喜欢喝拿铁，觉得美式咖啡太苦。...
[Step2-更新决策]      - [0.039] 项目进展与截止日期: 截至2025年6月26日，用户负责的AI项目进展还不错。在2025年6月25日，团队完成了数据预处理...
[Step2-更新决策]      - [0.031] 项目截止日期变更: 项目的截止日期从12月15日延期到1月5日，原因是后端需要更多时间

## Step 4: 查看处理结果

查看各阶段的详细结果

In [10]:
# 查看对话处理结果
conv_result = full_result["conversation_result"]

print("=" * 70)
print("📊 对话处理结果统计")
print("=" * 70)

print(f"\n1. 抽取的记忆: {len(conv_result['extracted_memories'])} 条")
for mem in conv_result['extracted_memories']:
    print(f"   - {mem.key}: {mem.value[:50]}...")

print(f"\n2. 更新决策: {len(conv_result['update_decisions'])} 条")
for decision in conv_result['update_decisions']:
    print(f"   - [{decision['action']}] {decision['candidate'].key}")
    if decision.get('reason'):
        print(f"     原因: {decision.get('reason', 'N/A')}")

print(f"\n3. 保存的记忆: {len(conv_result['saved_memories'])} 条")
for mem in conv_result['saved_memories']:
    print(f"   - ID: {mem.id[:8]}... | {mem.key}")

if conv_result['graph']:
    graph = conv_result['graph']
    print(f"\n4. 记忆图谱:")
    print(f"   - 节点数: {graph.get_node_count()}")
    print(f"   - 边数: {graph.get_edge_count()}")

if conv_result['errors']:
    print(f"\n⚠️ 错误: {conv_result['errors']}")

📊 对话处理结果统计

1. 抽取的记忆: 5 条
   - 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
   - 用户团队成员分工: 用户Tom的团队由五个人组成，分别负责不同任务：小李负责数据，小王负责前端。...
   - 用户对项目截止日期的担忧: 用户Tom担心12月15日的项目截止日期。...
   - 用户完成数据预处理模块: 用户Tom在2025年6月25日完成了数据预处理模块。...
   - 用户的饮品偏好: 用户Tom平时喜欢喝不加糖的美式咖啡。...

2. 更新决策: 5 条
   - [IGNORE] 用户负责AI项目的整体架构和模型训练
   - [ADD] 用户团队成员分工
   - [ADD] 用户对项目截止日期的担忧
   - [ADD] 用户完成数据预处理模块
   - [ADD] 用户的饮品偏好

3. 保存的记忆: 4 条
   - ID: 2f4716ef... | 用户团队成员分工
   - ID: 846177fe... | 用户对项目截止日期的担忧
   - ID: 2f0dbb05... | 用户完成数据预处理模块
   - ID: 94412bff... | 用户的饮品偏好

4. 记忆图谱:
   - 节点数: 4
   - 边数: 6


In [11]:
# 查看问答结果
print("=" * 70)
print("❓ 问答结果")
print("=" * 70)

for i, qa in enumerate(full_result["question_answers"], 1):
    print(f"\n问题 {i}: {qa['question']}")
    print("-" * 50)
    
    if qa['retrieved_memories']:
        print("检索到的记忆:")
        for item in qa['retrieved_memories']:
            mem = item['memory']
            score = item['score']
            print(f"  [{score:.3f}] {mem.key}: {mem.value[:40]}...")
    
    print(f"\n回答: {qa['answer']}")

❓ 问答结果

问题 1: Tom负责什么工作？
--------------------------------------------------
检索到的记忆:
  [0.034] 用户完成数据预处理模块: 用户Tom在2025年6月25日完成了数据预处理模块。...
  [0.028] 用户对项目截止日期的担忧: 用户Tom担心12月15日的项目截止日期。...
  [0.019] 用户的饮品偏好: 用户Tom平时喜欢喝不加糖的美式咖啡。...
  [0.005] 用户团队成员及分工: 用户Tom的团队由五人组成，成员包括小李负责数据，小王负责前端，用户负责整体架构...
  [-0.002] 用户的兴趣爱好: 用户Tom喜欢喝不加糖的美式咖啡。...

回答: Tom负责整体架构和模型训练。

问题 2: 项目的截止日期是什么时候？
--------------------------------------------------
检索到的记忆:
  [0.055] 用户团队成员分工: 用户Tom的团队由五个人组成，分别负责不同任务：小李负责数据，小王负责前端。...
  [0.009] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
  [0.004] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
  [0.001] 用户的饮品偏好: 用户Tom平时喜欢喝不加糖的美式咖啡。...
  [-0.002] 用户的兴趣爱好: 用户Tom喜欢喝不加糖的美式咖啡。...

回答: 项目的截止日期是2025年12月15日。

问题 3: 用户喜欢喝什么饮料？
--------------------------------------------------
检索到的记忆:
  [0.065] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
  [0.048] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
  [0.021] 用户的饮品偏好: 用户Tom平时喜欢喝不加糖的美式咖啡。...
  [0.006] 用户团队成员及分工: 用户Tom的团队由五人组成，成员包括小李负责数据，小王负责前端，用户负责整

## Step 5: 演示更新决策流程（第二轮对话）

模拟第二轮对话，展示更新模块如何处理与已有记忆的关系：
- **OVERWRITE**: 事实冲突（如截止日期变更）
- **MERGE**: 信息互补（如团队信息扩展）
- **VERSION**: 偏好/时间冲突（如咖啡偏好变更）

In [12]:
# 第二轮对话：触发各种更新场景
# 设计目标：
# 1. OVERWRITE - 事实冲突（截止日期变更）
# 2. MERGE - 信息补充（团队信息扩展）
# 3. VERSION - 偏好变更（咖啡偏好）
# 4. 冲突检测 - 团队人数变化

second_conversation = [
    # 场景1: 事实冲突 - 截止日期变更 (应触发 OVERWRITE)
    ConversationMessage(
        role="user",
        content="对了，我们的项目截止日期延期到了1月5日，因为后端需要更多时间。",
        timestamp="2025-06-27T10:00:00"
    ),
    ConversationMessage(
        role="assistant",
        content="好的，截止日期从12月15日延期到1月5日了。这样时间会宽裕一些。",
        timestamp="2025-06-27T10:01:00"
    ),
    
    # 场景2: 偏好变更 - 咖啡偏好改变 (应触发 VERSION 或 OVERWRITE)
    ConversationMessage(
        role="user",
        content="是的，另外我现在更喜欢喝拿铁了，美式太苦。",
        timestamp="2025-06-27T10:02:00"
    ),
    
    # 场景3: 信息补充 - 团队信息扩展 (应触发 MERGE)
    ConversationMessage(
        role="user",
        content="我们团队的小李是数据工程师，他之前在阿里工作过，经验很丰富。",
        timestamp="2025-06-27T10:03:00"
    ),
    
    # 场景4: 事实冲突 - 团队人数变化 (应触发冲突检测 + OVERWRITE)
    ConversationMessage(
        role="user",
        content="对了，团队现在有6个人了，新加入了一个测试工程师小张。",
        timestamp="2025-06-27T10:04:00"
    ),
]

print("第二轮对话（触发各种更新场景）:")
print("-" * 50)
print("设计的更新场景：")
print("  1. 截止日期变更: 12月15日 → 1月5日 (期望: OVERWRITE/事实冲突)")
print("  2. 咖啡偏好变更: 美式 → 拿铁 (期望: VERSION/偏好冲突)")
print("  3. 团队信息补充: 小李的背景 (期望: MERGE)")
print("  4. 团队人数变化: 5人 → 6人 (期望: OVERWRITE/事实冲突)")
print("-" * 50)
for msg in second_conversation:
    print(f"  [{msg.role}] {msg.content}")

第二轮对话（触发各种更新场景）:
--------------------------------------------------
设计的更新场景：
  1. 截止日期变更: 12月15日 → 1月5日 (期望: OVERWRITE/事实冲突)
  2. 咖啡偏好变更: 美式 → 拿铁 (期望: VERSION/偏好冲突)
  3. 团队信息补充: 小李的背景 (期望: MERGE)
  4. 团队人数变化: 5人 → 6人 (期望: OVERWRITE/事实冲突)
--------------------------------------------------
  [user] 对了，我们的项目截止日期延期到了1月5日，因为后端需要更多时间。
  [assistant] 好的，截止日期从12月15日延期到1月5日了。这样时间会宽裕一些。
  [user] 是的，另外我现在更喜欢喝拿铁了，美式太苦。
  [user] 我们团队的小李是数据工程师，他之前在阿里工作过，经验很丰富。
  [user] 对了，团队现在有6个人了，新加入了一个测试工程师小张。


In [13]:
# 处理第二轮对话，观察更新决策
print("\n处理第二轮对话，观察更新决策过程：")
print("=" * 70)

second_result = pipeline.process_conversation(second_conversation)

print("\n\n📋 第二轮更新决策详情:")
for i, decision in enumerate(second_result['update_decisions'], 1):
    print(f"\n决策 {i}:")
    print(f"  候选记忆: {decision['candidate'].key}")
    print(f"  操作: {decision['action']}")
    if decision.get('related'):
        print(f"  相关已有记忆: {len(decision['related'])} 条")
        for rm, score in decision['related']:
            print(f"    - [{score:.3f}] {rm.key}: {rm.value[:40]}...")
    if decision.get('deprecated_ids'):
        print(f"  废弃的旧记忆ID: {decision['deprecated_ids']}")


处理第二轮对话，观察更新决策过程：
[Step1-抽取] 开始从 5 条消息中抽取记忆...
[SimpleExtractor] 抽取了 4 条记忆
[Step1-抽取] 抽取到 4 条候选记忆
[Step2-更新决策] 开始对每条候选记忆进行更新决策...
[Step2-更新决策] 处理候选记忆 1/4: 项目截止日期延期至2025年1月5日
[Step2-更新决策]   → 找到 3 条相关记忆
[Step2-更新决策]      - [0.034] 用户对项目截止日期的担忧: 用户Tom担心12月15日的项目截止日期。...
[Step2-更新决策]      - [0.018] 用户完成数据预处理模块: 用户Tom在2025年6月25日完成了数据预处理模块。...
[Step2-更新决策]      - [0.017] 用户团队成员分工: 用户Tom的团队由五个人组成，分别负责不同任务：小李负责数据，小王负责前端。...
[MemoryUpdater] 决策: ADD - 与已有记忆相似度较低 (0.03)，作为新记忆添加
[Step2-更新决策]   → 决定: ADD
[Step2-更新决策] 处理候选记忆 2/4: 用户偏好饮品变化
[Step2-更新决策]   → 找到 3 条相关记忆
[Step2-更新决策]      - [0.070] 用户的饮品偏好: 用户Tom平时喜欢喝不加糖的美式咖啡。...
[Step2-更新决策]      - [0.044] 用户团队成员及分工: 用户Tom的团队由五人组成，成员包括小李负责数据，小王负责前端，用户负责整体架构和模型训练。...
[Step2-更新决策]      - [0.018] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
[MemoryUpdater] 决策: ADD - 与已有记忆相似度较低 (0.07)，作为新记忆添加
[Step2-更新决策]   → 决定: ADD
[Step2-更新决策] 处理候选记忆 3/4: 团队成员小李的背景
[Step2-更新决策]   → 找到 3 条相关记忆
[Step2-更新决策]      - [0.086] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
[Step2

In [14]:
# 再次回答问题，验证记忆已更新
print("\n验证更新后的记忆：")
print("=" * 70)

verification_questions = [
    "项目的截止日期是什么时候？",  # 应该回答1月5日
    "用户喜欢喝什么饮料？",         # 应该回答拿铁
    "团队有多少人？",              # 应该回答6人
    "小李是做什么的？"             # 应该回答数据工程师，阿里背景
]

for q in verification_questions:
    print(f"\n问题: {q}")
    qa_result = pipeline.answer_question(q)
    print(f"回答: {qa_result['answer']}")


验证更新后的记忆：

问题: 项目的截止日期是什么时候？
[Step5-检索] 检索与问题相关的记忆: 项目的截止日期是什么时候？
[Step5-检索]   → 找到 5 条相关记忆:
[Step5-检索]      - [0.076] 团队成员小李的背景: 用户知道团队中的数据工程师小李曾在阿里工作过，经验丰富。...
[Step5-检索]      - [0.055] 用户团队成员分工: 用户Tom的团队由五个人组成，分别负责不同任务：小李负责数据，小王负责前端。...
[Step5-检索]      - [0.015] 团队规模和新成员: 用户记得团队目前有6个人，最近新加入了一位测试工程师小张。...
[Step5-检索]      - [0.009] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
[Step5-检索]      - [0.004] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
[Step5-检索]   → 构建上下文完成
[Step5-回答] 调用LLM生成回答...
[Step5-回答]   → 回答: 项目的截止日期是2025年12月15日。...
回答: 项目的截止日期是2025年12月15日。

问题: 用户喜欢喝什么饮料？
[Step5-检索] 检索与问题相关的记忆: 用户喜欢喝什么饮料？
[Step5-检索]   → 找到 5 条相关记忆:
[Step5-检索]      - [0.065] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
[Step5-检索]      - [0.048] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
[Step5-检索]      - [0.021] 用户的饮品偏好: 用户Tom平时喜欢喝不加糖的美式咖啡。...
[Step5-检索]      - [0.006] 用户团队成员及分工: 用户Tom的团队由五人组成，成员包括小李负责数据，小王负责前端，用户负责整体架构和模型训练。...
[Step5-检索]      - [-0.002] 团队成员小李的背景: 用户知道团队中的数据工程师小李曾在阿里工作过，经验丰富。...
[St

## Step 5.5: 第三轮对话 - 演示IGNORE和更多冲突场景

添加第三轮对话来演示：
1. IGNORE - 重复信息（完全相同的记忆）
2. 更多的事实冲突
3. 时间冲突场景

In [15]:
# 第三轮对话：演示IGNORE和更多复杂场景
third_conversation = [
    # 场景1: IGNORE - 重复信息（与已有记忆完全相同）
    ConversationMessage(
        role="user",
        content="我叫Tom，我在负责一个AI项目。",  # 重复的基本信息
        timestamp="2025-06-28T09:00:00"
    ),
    
    # 场景2: 再次变更偏好 - 又改回来了（时间冲突）
    ConversationMessage(
        role="user",
        content="最近又开始喝美式了，拿铁太甜了。",
        timestamp="2025-06-28T09:01:00"
    ),
    
    # 场景3: 项目进展更新 - 补充信息（MERGE）
    ConversationMessage(
        role="user",
        content="项目进展很顺利，数据预处理和模型训练都完成了，现在在做集成测试。",
        timestamp="2025-06-28T09:02:00"
    ),
    
    # 场景4: 工作职责变更 - 事实冲突
    ConversationMessage(
        role="user",
        content="我现在主要负责模型优化和部署，架构设计交给小王了。",
        timestamp="2025-06-28T09:03:00"
    ),
    
    # 场景5: 新增关联信息 - 扩展已有记忆（MERGE）
    ConversationMessage(
        role="user",
        content="小李除了负责数据，现在还在帮忙做数据质量监控。",
        timestamp="2025-06-28T09:04:00"
    ),
]

print("第三轮对话（IGNORE和更多复杂场景）:")
print("-" * 50)
print("设计的更新场景：")
print("  1. 重复信息: Tom负责AI项目 (期望: IGNORE)")
print("  2. 偏好再次变更: 拿铁 → 美式 (期望: VERSION/时间冲突)")
print("  3. 项目进展更新: 补充完成情况 (期望: MERGE)")
print("  4. 工作职责变更: 架构→模型优化 (期望: OVERWRITE/事实冲突)")
print("  5. 小李职责扩展: 数据+监控 (期望: MERGE)")
print("-" * 50)
for msg in third_conversation:
    print(f"  [{msg.role}] {msg.content}")

第三轮对话（IGNORE和更多复杂场景）:
--------------------------------------------------
设计的更新场景：
  1. 重复信息: Tom负责AI项目 (期望: IGNORE)
  2. 偏好再次变更: 拿铁 → 美式 (期望: VERSION/时间冲突)
  3. 项目进展更新: 补充完成情况 (期望: MERGE)
  4. 工作职责变更: 架构→模型优化 (期望: OVERWRITE/事实冲突)
  5. 小李职责扩展: 数据+监控 (期望: MERGE)
--------------------------------------------------
  [user] 我叫Tom，我在负责一个AI项目。
  [user] 最近又开始喝美式了，拿铁太甜了。
  [user] 项目进展很顺利，数据预处理和模型训练都完成了，现在在做集成测试。
  [user] 我现在主要负责模型优化和部署，架构设计交给小王了。
  [user] 小李除了负责数据，现在还在帮忙做数据质量监控。


In [16]:
# 处理第三轮对话，观察更新决策
print("\n处理第三轮对话，观察更新决策过程：")
print("=" * 70)

third_result = pipeline.process_conversation(third_conversation)

print("\n\n📋 第三轮更新决策详情:")
for i, decision in enumerate(third_result['update_decisions'], 1):
    print(f"\n决策 {i}:")
    print(f"  候选记忆: {decision['candidate'].key}")
    print(f"  候选内容: {decision['candidate'].value[:60]}...")
    print(f"  操作: {decision['action']}")
    if decision.get('related'):
        print(f"  相关已有记忆: {len(decision['related'])} 条")
        for rm, score in decision['related']:
            print(f"    - [{score:.3f}] {rm.key}: {rm.value[:40]}...")
    if decision.get('deprecated_ids'):
        print(f"  废弃的旧记忆ID: {decision['deprecated_ids']}")


处理第三轮对话，观察更新决策过程：
[Step1-抽取] 开始从 5 条消息中抽取记忆...
[SimpleExtractor] 抽取了 3 条记忆
[Step1-抽取] 抽取到 3 条候选记忆
[Step2-更新决策] 开始对每条候选记忆进行更新决策...
[Step2-更新决策] 处理候选记忆 1/3: 用户的身份和项目职责
[Step2-更新决策]   → 找到 3 条相关记忆
[Step2-更新决策]      - [0.071] 项目截止日期延期至2025年1月5日: 用户记得他们的项目截止日期从原本的12月15日延期到了2025年1月5日，因为后端开发需要更多时间。...
[Step2-更新决策]      - [0.033] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
[Step2-更新决策]      - [0.030] 用户负责AI项目的整体架构和模型训练: 用户Tom负责AI项目的整体架构设计和模型训练工作。...
[MemoryUpdater] 决策: ADD - 与已有记忆相似度较低 (0.07)，作为新记忆添加
[Step2-更新决策]   → 决定: ADD
[Step2-更新决策] 处理候选记忆 2/3: 用户的饮品偏好变化
[Step2-更新决策]   → 找到 3 条相关记忆
[Step2-更新决策]      - [0.047] 项目截止日期: 用户Tom担心2025年12月15日的项目截止日期。...
[Step2-更新决策]      - [0.038] 团队成员小李的背景: 用户知道团队中的数据工程师小李曾在阿里工作过，经验丰富。...
[Step2-更新决策]      - [0.020] 用户偏好饮品变化: 用户现在更喜欢喝拿铁咖啡，觉得美式咖啡太苦了。...
[MemoryUpdater] 决策: ADD - 与已有记忆相似度较低 (0.05)，作为新记忆添加
[Step2-更新决策]   → 决定: ADD
[Step2-更新决策] 处理候选记忆 3/3: 项目进展情况
[Step2-更新决策]   → 找到 3 条相关记忆
[Step2-更新决策]      - [0.048] 用户的兴趣爱好: 用户Tom喜欢喝不加糖的美式咖啡。...
[Step2-更新决策]      - [0

In [17]:
# 汇总所有轮次的更新决策统计
print("\n" + "=" * 70)
print("📊 所有轮次更新决策统计汇总")
print("=" * 70)

all_decisions = []
all_decisions.extend([(1, d) for d in full_result["conversation_result"]["update_decisions"]])
all_decisions.extend([(2, d) for d in second_result["update_decisions"]])
all_decisions.extend([(3, d) for d in third_result["update_decisions"]])

# 统计各类操作
action_counts = {}
for round_num, decision in all_decisions:
    action = decision["action"]
    action_counts[action] = action_counts.get(action, 0) + 1

print("\n操作类型统计:")
for action, count in sorted(action_counts.items()):
    desc = {
        "ADD": "新增记忆（无相关已有记忆）",
        "MERGE": "合并记忆（信息互补）",
        "OVERWRITE": "覆盖记忆（事实冲突）",
        "VERSION": "版本化（偏好/时间冲突）",
        "IGNORE": "忽略（重复信息）"
    }.get(action, "未知操作")
    print(f"  {action}: {count} 次 - {desc}")

print("\n详细决策列表:")
for round_num, decision in all_decisions:
    action_icon = {
        "ADD": "➕",
        "MERGE": "🔀",
        "OVERWRITE": "🔄",
        "VERSION": "📝",
        "IGNORE": "⏭️"
    }.get(decision["action"], "❓")
    print(f"  [轮次{round_num}] {action_icon} {decision['action']}: {decision['candidate'].key}")


📊 所有轮次更新决策统计汇总

操作类型统计:
  ADD: 11 次 - 新增记忆（无相关已有记忆）
  IGNORE: 1 次 - 忽略（重复信息）

详细决策列表:
  [轮次1] ⏭️ IGNORE: 用户负责AI项目的整体架构和模型训练
  [轮次1] ➕ ADD: 用户团队成员分工
  [轮次1] ➕ ADD: 用户对项目截止日期的担忧
  [轮次1] ➕ ADD: 用户完成数据预处理模块
  [轮次1] ➕ ADD: 用户的饮品偏好
  [轮次2] ➕ ADD: 项目截止日期延期至2025年1月5日
  [轮次2] ➕ ADD: 用户偏好饮品变化
  [轮次2] ➕ ADD: 团队成员小李的背景
  [轮次2] ➕ ADD: 团队规模和新成员
  [轮次3] ➕ ADD: 用户的身份和项目职责
  [轮次3] ➕ ADD: 用户的饮品偏好变化
  [轮次3] ➕ ADD: 项目进展情况


In [ ]:
# 最终验证：回答问题检验记忆更新效果
print("\n" + "=" * 70)
print("🔍 最终验证：回答问题检验记忆更新效果")
print("=" * 70)

final_questions = [
    "Tom现在负责什么工作？",          # 应该是模型优化和部署
    "项目截止日期是什么时候？",        # 应该是1月5日
    "用户喜欢喝什么饮料？",            # 应该是美式（最新）
    "团队有多少人？分别是谁？",        # 应该是6人
    "小李负责什么工作？",              # 应该是数据+监控
    "项目进展如何？"                   # 应该包含集成测试
]

for q in final_questions:
    print(f"\n❓ 问题: {q}")
    qa_result = pipeline.answer_question(q)
    print(f"💬 回答: {qa_result['answer']}")

## Step 6: 查看存储状态

验证Neo4j和Milvus中的数据同步

In [ ]:
# 查看存储状态
store = pipeline.store

print("=" * 70)
print("💾 存储状态")
print("=" * 70)

# 获取所有用户记忆
all_memories = store.get_all(user_id=USER_ID)
print(f"\n用户 {USER_ID} 的所有记忆: {len(all_memories)} 条")

for i, mem in enumerate(all_memories, 1):
    status_icon = "✓" if mem.status == "activated" else "○"
    print(f"  {status_icon} {i}. [{mem.id[:8]}...] {mem.key}")
    print(f"       内容: {mem.value[:50]}...")
    print(f"       状态: {mem.status}, 类型: {mem.memory_type}")
    print(f"       创建: {mem.created_at}")

In [ ]:
# 验证向量检索功能
print("\n验证向量检索功能:")
print("-" * 50)

test_query = "项目进展"
search_results = store.search_similar(test_query, top_k=3, user_id=USER_ID)

print(f"查询: '{test_query}'")
print(f"结果: {len(search_results)} 条")
for mem, score in search_results:
    print(f"  [{score:.3f}] {mem.key}: {mem.value[:40]}...")

## 总结

本notebook演示了记忆工程的**完整串行流程**和**多轮对话场景下的更新决策**：

### 正确的流程顺序

```
对话输入
    ↓
[1. 记忆抽取] MemoryExtractor
    - 从对话中识别有价值的信息
    - 输出候选记忆列表
    ↓
[2. 更新决策] MemoryUpdater.decide_action()
    - 查询已有记忆（调用MemoryStore.find_related_memories）
    - 决定操作：ADD / MERGE / OVERWRITE / VERSION / IGNORE
    - 处理冲突和合并
    ↓
[3. 统一存储] MemoryStore.save()
    - 生成Embedding
    - 同步写入Neo4j（结构化数据）
    - 同步写入Milvus（向量数据）
    - 确保ID对应
    ↓
[4. 记忆组织] MemoryOrganizer
    - 构建时间/事件/语义结构
    - 建立记忆之间的关系边
    ↓
[5. 检索回答] MemoryStore.search_similar() + LLM
    - 向量检索相关记忆
    - 注入上下文
    - LLM生成回答
```

### 更新决策类型演示

本notebook通过三轮对话演示了所有更新决策类型：

| 操作类型 | 触发条件 | 演示场景 |
|---------|---------|---------|
| **ADD** | 无相关已有记忆 | 第一轮对话的所有记忆 |
| **MERGE** | 相似度0.6-0.85，无冲突 | 团队信息扩展、小李职责补充 |
| **OVERWRITE** | 相似度0.6-0.85，事实冲突 | 截止日期变更、团队人数变化、工作职责变更 |
| **VERSION** | 相似度0.6-0.85，偏好/时间冲突 | 咖啡偏好多次变更 |
| **IGNORE** | 相似度>0.85，内容完全相同 | 重复的基本信息 |

### 关键设计点

1. **统一存储管理器** (`MemoryStore`)：确保Neo4j和Milvus的ID同步
2. **更新决策前置**：先查询相关记忆，再决定如何处理新记忆
3. **流程可控**：各模块不自动保存，由Pipeline统一管理
4. **冲突检测**：使用LLM判断事实冲突、偏好冲突、时间冲突

In [ ]:
print("\n" + "=" * 70)
print("🎉 记忆工程完整串行流程演示完成！")
print("=" * 70)
print("""
流程回顾：
1. ✓ 记忆抽取 - 从对话中抽取候选记忆
2. ✓ 更新决策 - 查询相关记忆，决定ADD/MERGE/OVERWRITE/VERSION/IGNORE
3. ✓ 统一存储 - Neo4j + Milvus同步写入，ID对应
4. ✓ 记忆组织 - 构建图谱结构
5. ✓ 检索回答 - 向量检索 + LLM回答

更新决策演示：
- ➕ ADD: 新增记忆（无相关已有记忆）
- 🔀 MERGE: 合并记忆（信息互补）
- 🔄 OVERWRITE: 覆盖记忆（事实冲突）
- 📝 VERSION: 版本化（偏好/时间冲突）
- ⏭️ IGNORE: 忽略（重复信息）
""")